# Epithelial Cell BBKNN Integration Pipeline - Optimized v1.3

Complete BBKNN batch correction workflow with:
1. Complete BBKNN batch correction workflow with post-hoc parameter validation
2. Force retention of specified markers (avoid removal in HVG/filtering)
3. Differential analysis/plotting defaults to reading from .raw full gene matrix (use_raw=True)
4. Multi-resolution Leiden clustering; UMAP uses BBKNN-constructed neighbor graph
5. Detailed and reproducible visualization and reporting

Author: Clinical-Bioinformatics Team  
Version: v1.3

## Import Libraries

In [ ]:
import sys
import os
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from scipy.sparse import issparse, csr_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import time
from tqdm import tqdm
import pickle
warnings.filterwarnings('ignore')

## Configuration Section

In [ ]:
# ---------- Input/Output ----------
INPUT_H5AD_PATH = "/home/h2048/data/py/1128/bbknn_celltype_analysis/Epithelial/adata_Epithelial_bbknn.h5ad"
OUTPUT_DIR = "/home/h2048/data/py/1128/bbknn_celltype_analysis/Epithelial/output_optimized"
OVERWRITE_EXISTING = True

# ---------- BBKNN Integration Parameters ----------
BATCH_KEY = "dataset"
BBKNN_NEIGHBORS_WITHIN_BATCH = 3
BBKNN_N_PCS = 50

# ---------- High Variable Genes (HVG) ----------
USE_HVG = True
N_TOP_GENES = 3000
HVG_FLAVOR = "seurat_v3"

# ---------- Gene Filtering ----------
MIN_CELLS_PER_GENE = 3

# ---------- Raw Counts Source ----------
RAW_COUNTS_SOURCE = "auto"

# ---------- Normalization ----------
NORMALIZE_TOTAL = True
TARGET_SUM = 1e4
LOG_TRANSFORM = True
SCALE_DATA = True
MAX_VALUE = 10

# ---------- PCA ----------
N_PCS = 50

# ---------- Dimensionality Reduction/Visualization ----------
RUN_UMAP = True
UMAP_MIN_DIST = 0.5

# ---------- Clustering ----------
RUN_CLUSTERING = True
LEIDEN_RESOLUTIONS = [1.2, 1.6, 2.0, 2.4, 2.8]
DEFAULT_RESOLUTION = 2.0

In [ ]:
# ---------- Epithelial Cell Marker Genes ----------
MARKER_EPITHELIAL = [
    # Basal
    'TP63','KRT5','KRT14',
    # Secretory
    'SCGB1A1','SERPINB3','SCGB3A2','SCGB3A1','TCN1','ASRGL1',
    # Ciliated
    'FOXJ1','RSPH1','PIFO','BEST4','C20orf85','C9orf24',
    # Differentiation
    'KRT19','NOTCH3','KRT16','KRT23',
    # Goblet
    'MUC5AC','SPDEF','LYPD2','ITLN1',
    # Neuroendocrine
    'ASCL1','GRP','KRT8',
    # Tuft / Ionocytes
    'POU2F3','ASCL2','CFTR','FOXI1','ASCL3','BSND','IGF1','CLCNKB','PDE1C',
    # AT1
    'AGER','RTKN2','CLIC5','SPOCK2','TIMP3',
    # AT2
    'SFTPC','LAMP3','MFSD2A','C8orf4','C11orf96','SFTPB','SFTA2',
    # Mesenchymal
    'VIM','SOX9','MYH11','ACTA2','MYLK',
    # Secretory/Immune
    'DMBT1','RNASE1','MUC5B','LYZ','LTF','PIP','CCL28',
    # Ciliogenesis
    'DEUP1','FOXN4','CDC20B','CCNO',
    # Proliferation
    'MKI67','TOP2A','TK1','CENPW'
]

# Key markers for cell typing/visualization
KEY_MARKERS = {
    'Basal': ['TP63','KRT5'],
    'Secretory': ['SCGB1A1','SCGB3A2'],
    'Ciliated': ['FOXJ1','RSPH1'],
    'Goblet': ['MUC5AC','SPDEF'],
    'Tuft': ['POU2F3','ASCL2'],
    'Ionocyte': ['FOXI1','CFTR'],
    'AT1': ['AGER','RTKN2'],
    'AT2': ['SFTPC','SFTPB'],
    'Proliferating': ['MKI67','TOP2A']
}

In [ ]:
# ---------- Marker Gene Analysis ----------
RUN_FIND_MARKERS = True
MARKER_MIN_PCT = 0.25
MARKER_LOGFC_THRESHOLD = 0.25
TOP_N_MARKERS = 10

# ---------- Performance/Caching ----------
USE_CACHE = True
CHUNK_SIZE = 100

# ---------- Visualization ----------
GENERATE_DOTPLOT = True
GENERATE_HEATMAP = True
GENERATE_FACET_PLOTS = True
DPI = 300
FIGURE_FORMAT = "png"

VERBOSE = True

# ---------- Force Include Markers ----------
FORCE_INCLUDE_MARKERS = sorted({
    *MARKER_EPITHELIAL,
    *[g for genes in KEY_MARKERS.values() for g in genes],
})

## Utility Functions

In [ ]:
def log_msg(msg):
    """Lightweight logging (controlled by VERBOSE)."""
    if VERBOSE:
        print(msg)

def log_step(step_num, step_name):
    """Print step headers in uniform format."""
    log_msg("\n" + "="*70)
    log_msg(f"Step {step_num}: {step_name}")
    log_msg("="*70)

def save_checkpoint(data, filename, output_dir):
    """Save intermediate results to cache file (when USE_CACHE is enabled)."""
    if USE_CACHE:
        checkpoint_path = Path(output_dir) / f".cache_{filename}"
        with open(checkpoint_path, 'wb') as f:
            pickle.dump(data, f)
        log_msg(f"   Checkpoint saved: {checkpoint_path}")

def load_checkpoint(filename, output_dir):
    """Load cache file (if exists and USE_CACHE is enabled)."""
    if USE_CACHE:
        checkpoint_path = Path(output_dir) / f".cache_{filename}"
        if checkpoint_path.exists():
            with open(checkpoint_path, 'rb') as f:
                return pickle.load(f)
    return None

## Main Analysis Functions

In [ ]:
def load_anndata(h5ad_path):
    """Load AnnData and check batch column and distribution."""
    import scanpy as sc
    log_step(1, "Load Data")
    log_msg(f"\nReading file: {h5ad_path}")
    if not Path(h5ad_path).exists():
        raise FileNotFoundError(f"File not found: {h5ad_path}")

    adata = sc.read_h5ad(h5ad_path)
    log_msg(f"Data loaded successfully")
    log_msg(f"   Cells: {adata.n_obs:,}")
    log_msg(f"   Genes: {adata.n_vars:,}")

    log_msg(f"\nAvailable metadata columns:")
    for col in adata.obs.columns:
        n_unique = adata.obs[col].nunique()
        log_msg(f"   - {col}: {n_unique} unique values")

    if BATCH_KEY not in adata.obs.columns:
        raise ValueError(f"Batch column '{BATCH_KEY}' not found in adata.obs")

    log_msg(f"\nBatch distribution (key: {BATCH_KEY}):")
    batch_counts = adata.obs[BATCH_KEY].value_counts().sort_index()
    for batch, count in batch_counts.items():
        pct = count / adata.n_obs * 100
        log_msg(f"   {batch}: {count:,} cells ({pct:.1f}%)")
    return adata

In [ ]:
def check_marker_genes(adata):
    """Check marker availability (both var and raw)."""
    log_step(2, "Check Marker Genes")
    raw_names = set(adata.raw.var_names) if adata.raw is not None else set()
    var_names = set(adata.var_names)

    def present(g):
        return (g in var_names) or (g in raw_names)

    available_markers = [g for g in MARKER_EPITHELIAL if present(g)]
    missing_markers   = [g for g in MARKER_EPITHELIAL if not present(g)]

    log_msg(f"\nMarker gene availability:")
    log_msg(f"   Total: {len(MARKER_EPITHELIAL)}")
    log_msg(f"   Available: {len(available_markers)} ({len(available_markers)/len(MARKER_EPITHELIAL)*100:.1f}%)")
    log_msg(f"   Missing: {len(missing_markers)} ({len(missing_markers)/len(MARKER_EPITHELIAL)*100:.1f}%)")

    if missing_markers:
        log_msg(f"\n   Missing genes (sample): {', '.join(missing_markers[:10])}")
        if len(missing_markers) > 10:
            log_msg(f"   ... and {len(missing_markers)-10} more")

    log_msg(f"\nKey cell type markers:")
    for celltype, genes in KEY_MARKERS.items():
        available = [g for g in genes if present(g)]
        log_msg(f"   {celltype}: {len(available)}/{len(genes)} available")
    return available_markers

In [ ]:
def preprocess_for_bbknn(adata, n_hvg=3000, batch_key=None, FORCE_INCLUDE_MARKERS=None):
    """
    Purpose:
      - Save raw counts to layers['counts']
      - Save normalized + log1p expression to layers['log1p']
      - Return (adata_full, adata_hvg)

    Description:
      - adata_full: Full gene matrix, X=log1p-normalized, layers['counts']=raw counts, layers['log1p']=same as X
      - adata_hvg: HVG (+force retained) subset, X=log1p-normalized, layers['counts']=subset raw counts
      - adata_hvg.raw = adata_full: Keep full genes in .raw for downstream diff analysis/visualization (use_raw=True)
    """
    import scanpy as sc
    
    def _log(msg):
        try:
            log_msg(msg)
        except NameError:
            print(msg)

    _log("="*70)
    _log(f"Select HVGs (n={n_hvg}) and set layers, without directly writing to .raw")
    _log("-"*70)

    # 1) Ensure raw counts in layers['counts']
    if 'counts' in adata.layers:
        counts = adata.layers['counts']
        _log("   Using existing layers['counts'] as raw counts")
    elif getattr(adata, "raw", None) is not None:
        counts = adata.raw.X
        _log("   Extracting raw counts from .raw -> layers['counts']")
    else:
        counts = adata.X
        _log("   No .raw detected, using current X as raw counts -> layers['counts']")

    if not issparse(counts):
        counts = csr_matrix(counts)
    adata.layers['counts'] = counts

    # 2) HVG selection (with robust fallback)
    adata.X = adata.layers['counts'].copy()

    try:
        sc.pp.highly_variable_genes(
            adata,
            n_top_genes=n_hvg,
            flavor="seurat_v3",
            batch_key=batch_key
        )
        _log(f"   HVG(seurat_v3, batch_key={batch_key}) successful")
    except Exception as e:
        _log(f"   ⚠️  HVG(seurat_v3, batch_key={batch_key}) failed: {repr(e)}")
        _log("      → Fallback to seurat_v3 without batch_key")
        sc.pp.highly_variable_genes(
            adata,
            n_top_genes=n_hvg,
            flavor="seurat_v3",
            batch_key=None
        )

    hv_mask = adata.var['highly_variable'].to_numpy(dtype=bool)

    # Force retain markers
    if FORCE_INCLUDE_MARKERS is None:
        FORCE_INCLUDE_MARKERS = []
    force_mask = adata.var_names.isin(FORCE_INCLUDE_MARKERS)

    keep_mask = hv_mask | force_mask
    n_force = int(force_mask.sum())
    _log(f"   Selected HVGs: {int(hv_mask.sum())}; Force retained: {n_force}; Total: {int(keep_mask.sum())}")

    # 3) Normalize on full matrix to get adata_full (X=log1p)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.layers['log1p'] = adata.X.copy()

    # 4) Create HVG subset object (without overwriting .raw)
    adata_hvg = adata[:, keep_mask].copy()
    adata_hvg.layers['counts'] = adata.layers['counts'][:, keep_mask].copy()

    # Make HVG subset X = log1p of subset counts
    adata_hvg.X = adata_hvg.layers['counts'].copy()
    sc.pp.normalize_total(adata_hvg, target_sum=1e4)
    sc.pp.log1p(adata_hvg)
    adata_hvg.layers['log1p'] = adata_hvg.X.copy()

    # 5) Record basic metadata
    adata_hvg.uns['hvg_n_top_genes'] = int(n_hvg)
    adata_hvg.uns['force_include_markers'] = np.array(FORCE_INCLUDE_MARKERS, dtype=str)
    adata_hvg.uns['keep_mask_sum'] = int(keep_mask.sum())

    # 6) Set full gene data as HVG subset's .raw (for use_raw=True access)
    adata_hvg.raw = adata
    _log(f"   Set adata_hvg.raw = adata_full (contains {adata.n_vars} genes)")

    _log("   ✓ Complete: raw counts in layers['counts'], log1p in layers['log1p']")
    _log("="*70)

    return adata, adata_hvg

In [ ]:
def run_pca(adata, n_pcs=50):
    """Run PCA dimensionality reduction."""
    import scanpy as sc
    log_msg(f"\nRunning PCA (n_comps={n_pcs})...")
    sc.tl.pca(adata, n_comps=n_pcs, svd_solver='arpack')
    var_ratio = adata.uns['pca']['variance_ratio']
    cumsum_var = np.cumsum(var_ratio)
    if len(cumsum_var) >= 50:
        log_msg(f"   PC1-10 explained variance: {cumsum_var[9]:.2%}")
        log_msg(f"   PC1-20 explained variance: {cumsum_var[19]:.2%}")
        log_msg(f"   PC1-50 explained variance: {cumsum_var[49]:.2%}")
    return adata

In [ ]:
def _bbknn_sanity_check(adata):
    """Print post-hoc validation info for BBKNN neighbor counts."""
    try:
        n_batches = adata.obs[BATCH_KEY].nunique()
        expected = BBKNN_NEIGHBORS_WITHIN_BATCH * n_batches
        neigh = adata.uns.get('neighbors', {}).get('params', {})
        actual = neigh.get('n_neighbors', None)
        log_msg(
            f"   BBKNN check → n_batches={n_batches}, "
            f"neighbors_within_batch={BBKNN_NEIGHBORS_WITHIN_BATCH} → "
            f"expected total neighbors≈{expected}, actual {actual}"
        )
    except Exception as e:
        log_msg(f"   ⚠️ BBKNN post-hoc validation failed: {e}")


def run_bbknn_integration(adata, batch_key='dataset', neighbors_within_batch=3, n_pcs=50):
    """Run BBKNN batch integration."""
    from bbknn import bbknn as bbknn_func
    log_step(4, "BBKNN Batch Integration")
    log_msg("\nBBKNN parameters:")
    log_msg(f"   batch_key: {batch_key}")
    log_msg(f"   neighbors_within_batch: {neighbors_within_batch}")
    log_msg(f"   n_pcs: {n_pcs}")

    log_msg("\nRunning BBKNN...")
    start_time = time.time()
    bbknn_func(
        adata,
        batch_key=batch_key,
        neighbors_within_batch=neighbors_within_batch,
        n_pcs=n_pcs,
        copy=False
    )
    elapsed_time = time.time() - start_time
    log_msg(f"\n   BBKNN complete, time elapsed: {elapsed_time:.1f} seconds")

    _bbknn_sanity_check(adata)
    return adata

In [ ]:
def run_umap_and_clustering(adata, run_umap=True, min_dist=0.3,
                            run_clustering=True, resolutions=[1.2, 1.6, 2.0, 2.4, 2.8],
                            default_resolution=2.0):
    """Run UMAP using BBKNN neighbor graph and multi-resolution Leiden clustering."""
    import scanpy as sc
    log_step(5, "UMAP and Multi-resolution Clustering")

    if run_umap:
        log_msg(f"\nRunning UMAP (min_dist={min_dist}, using BBKNN neighbor graph)...")
        sc.tl.umap(adata, min_dist=min_dist)
        log_msg("   UMAP complete")

    if run_clustering:
        log_msg(f"\nRunning multi-resolution Leiden clustering...")
        log_msg(f"   Resolution list: {resolutions}")
        for res in resolutions:
            cluster_key = f'leiden_bbknn_res{res}'
            try:
                sc.tl.leiden(adata, resolution=res, key_added=cluster_key, flavor='igraph',
                             n_iterations=2, directed=False)
            except TypeError:
                sc.tl.leiden(adata, resolution=res, key_added=cluster_key, n_iterations=2)
            n_clusters = adata.obs[cluster_key].nunique()
            log_msg(f"   Resolution {res}: {n_clusters} clusters")
        default_key = f'leiden_bbknn_res{default_resolution}'
        adata.obs['leiden_bbknn'] = adata.obs[default_key]
        log_msg(f"\n   Default clustering: {default_key}")
    return adata

In [ ]:
def compute_pct_expressed_vectorized(adata, cluster_key, cluster, genes):
    """Vectorized computation of gene expression percentage in/out of cluster; prefer var, fallback to raw."""
    cluster_mask = (adata.obs[cluster_key] == cluster).values
    n_in = int(cluster_mask.sum()); n_out = int((~cluster_mask).sum())
    if n_in == 0 or n_out == 0 or len(genes) == 0:
        return [], []

    genes_in_var = [g for g in genes if g in adata.var_names]
    rem = [g for g in genes if g not in adata.var_names]
    X_in = X_out = None

    if genes_in_var:
        Xin = adata[cluster_mask, genes_in_var].X
        Xout = adata[~cluster_mask, genes_in_var].X
        Xin = Xin.toarray() if hasattr(Xin, 'toarray') else np.asarray(Xin)
        Xout = Xout.toarray() if hasattr(Xout, 'toarray') else np.asarray(Xout)
        X_in, X_out = Xin, Xout

    if rem and (adata.raw is not None):
        raw_names = np.array(adata.raw.var_names)
        raw_map = {g:i for i,g in enumerate(raw_names)}
        rem_exist = [g for g in rem if g in raw_map]
        if rem_exist:
            ridx = [raw_map[g] for g in rem_exist]
            R = adata.raw.X
            Rin = R[cluster_mask][:, ridx]
            Rout = R[~cluster_mask][:, ridx]
            Rin = Rin.toarray() if hasattr(Rin, 'toarray') else np.asarray(Rin)
            Rout = Rout.toarray() if hasattr(Rout, 'toarray') else np.asarray(Rout)
            if X_in is None:
                X_in, X_out = Rin, Rout
                genes_in_var = rem_exist
            else:
                X_in = np.concatenate([X_in, Rin], axis=1)
                X_out = np.concatenate([X_out, Rout], axis=1)
                genes_in_var = genes_in_var + rem_exist

    if X_in is None:
        return [], []
    pct_in  = (X_in  > 0).sum(axis=0) / n_in
    pct_out = (X_out > 0).sum(axis=0) / n_out
    return pct_in.tolist(), pct_out.tolist()

In [ ]:
def find_all_markers_optimized(adata, cluster_key, output_dir,
                               min_pct=0.25,
                               logfc_threshold=0.25):
    """
    Optimized FindAllMarkers:
      - Safe return when results are empty
      - Control export limit per cluster (TOP_N_MARKERS)
      - Differential analysis uses .raw full gene matrix (use_raw=True)
    """
    import scanpy as sc
    log_step(6, "Compute Cluster Differential Genes (Optimized)")

    cache_file = "epithelial_markers_cache.pkl"
    cached_markers = load_checkpoint(cache_file, output_dir)
    if cached_markers is not None:
        log_msg("\n   Loading from cache...")
        log_msg(f"   Total markers: {len(cached_markers)}")
        return cached_markers

    markers_file = Path(output_dir) / "cluster_markers.csv"

    log_msg(f"\nRunning differential analysis (Wilcoxon)...")
    log_msg(f"   min_pct: {min_pct}")
    log_msg(f"   logfc_threshold: {logfc_threshold}")

    log_msg("\n   Computing differentials...")
    sc.tl.rank_genes_groups(
        adata,
        groupby=cluster_key,
        method='wilcoxon',
        key_added='rank_genes_groups',
        use_raw=True,
        layer=None
    )
    log_msg("   ✓ Differential analysis complete")

    log_msg("\n   Extracting and filtering differential results...")
    clusters = adata.obs[cluster_key].unique()
    markers_list = []

    for cluster in tqdm(clusters, desc="Processing clusters"):
        df = sc.get.rank_genes_groups_df(adata, group=cluster)
        df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["logfoldchanges","pvals_adj"])
        df = df[(df["logfoldchanges"] > logfc_threshold) & (df["pvals_adj"] < 0.05)].copy()
        if df.empty:
            continue

        genes = df["names"].tolist()
        pct_in, pct_out = compute_pct_expressed_vectorized(adata, cluster_key, cluster, genes)
        if len(pct_in) != len(genes):
            n = len(genes)
            pct_in  = (pct_in  + [np.nan]*(n-len(pct_in)))[:n]
            pct_out = (pct_out + [np.nan]*(n-len(pct_out)))[:n]

        df["cluster"] = cluster
        df["pct_in_cluster"] = pct_in
        df["pct_out_cluster"] = pct_out
        df = df[df["pct_in_cluster"] > min_pct]
        if df.empty:
            continue

        if TOP_N_MARKERS is not None and TOP_N_MARKERS > 0:
            df = df.sort_values("pvals_adj", ascending=True).head(TOP_N_MARKERS)
        markers_list.append(df)

    if len(markers_list) == 0:
        log_msg("\n   ⚠️ No differential genes meet current thresholds, returning empty results.")
        empty_cols = ["names","scores","logfoldchanges","pvals","pvals_adj","cluster","pct_in_cluster","pct_out_cluster"]
        empty_df = pd.DataFrame(columns=empty_cols)
        empty_df.to_csv(markers_file, index=False)
        save_checkpoint(empty_df, cache_file, output_dir)
        return empty_df

    all_markers = pd.concat(markers_list, ignore_index=True)
    log_msg(f"\n   ✓ Total {len(all_markers)} markers obtained, covering {len(clusters)} clusters")
    save_checkpoint(all_markers, cache_file, output_dir)
    all_markers.to_csv(markers_file, index=False)
    log_msg(f"   ✓ Saved: {markers_file}")
    return all_markers

In [ ]:
def generate_visualizations(adata, available_markers, output_dir):
    """Generate UMAP overview, multi-resolution clustering, marker UMAPs, Dotplot, Heatmap, batch facets."""
    import scanpy as sc
    log_step(7, "Generate Visualizations")

    fig_dir = Path(output_dir) / "figures"
    fig_dir.mkdir(exist_ok=True)
    sc.settings.figdir = fig_dir

    # 1) Overview plot
    log_msg("\n   Generating integration quality overview...")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    sc.pl.umap(adata, color=BATCH_KEY, ax=axes[0], show=False, title='Batch')
    sc.pl.umap(adata, color='leiden_bbknn', ax=axes[1], show=False,
               title='Clusters', legend_loc='on data', legend_fontsize=8)
    if 'cell_type' in adata.obs.columns:
        sc.pl.umap(adata, color='cell_type', ax=axes[2], show=False, title='Cell Type')
    else:
        axes[2].axis('off')
    plt.tight_layout()
    plt.savefig(fig_dir / f'umap_overview.{FIGURE_FORMAT}', dpi=DPI, bbox_inches='tight')
    plt.close()
    log_msg(f"   ✓ Overview UMAP saved")

    # 2) Multi-resolution clustering
    if RUN_CLUSTERING and len(LEIDEN_RESOLUTIONS) > 1:
        log_msg("\n   Generating multi-resolution clustering plots...")
        n_res = len(LEIDEN_RESOLUTIONS)
        n_cols = 3
        n_rows = (n_res + n_cols - 1) // n_cols
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 6*n_rows))
        axes = axes.flatten() if n_res > 1 else [axes]
        for i, res in enumerate(LEIDEN_RESOLUTIONS):
            cluster_key = f'leiden_bbknn_res{res}'
            sc.pl.umap(
                adata, color=cluster_key, ax=axes[i], show=False,
                title=f'Resolution {res}', legend_loc='on data', legend_fontsize=8
            )
        for i in range(n_res, len(axes)):
            axes[i].axis('off')
        plt.tight_layout()
        plt.savefig(fig_dir / f'umap_resolutions.{FIGURE_FORMAT}', dpi=DPI, bbox_inches='tight')
        plt.close()
        log_msg(f"   ✓ Multi-resolution UMAP saved")

    # 3) Key marker expression UMAPs (use_raw=True)
    log_msg("\n   Generating marker expression UMAPs...")
    for celltype, genes in KEY_MARKERS.items():
        available = [g for g in genes if (g in (adata.raw.var_names if adata.raw is not None else adata.var_names))]
        if available:
            sc.pl.umap(
                adata,
                color=available,
                use_raw=True,
                cmap='Reds',
                ncols=len(available),
                vmax='p99',
                save=f'_{celltype}_markers.{FIGURE_FORMAT}'
            )
            log_msg(f"   ✓ {celltype} markers UMAP saved")

    # 4) Dotplot (use_raw=True)
    if GENERATE_DOTPLOT and len(available_markers) > 0:
        log_msg("\n   Generating marker Dotplot...")
        try:
            markers_to_plot = available_markers[:50] if len(available_markers) > 50 else available_markers
            sc.pl.dotplot(
                adata,
                var_names=markers_to_plot,
                groupby='leiden_bbknn',
                standard_scale='var',
                use_raw=True,
                save=f'_epithelial_markers.{FIGURE_FORMAT}',
                figsize=(max(20, len(markers_to_plot)*0.4), 10)
            )
            log_msg(f"   ✓ Dotplot saved (showing {len(markers_to_plot)} genes)")
        except Exception as e:
            log_msg(f"   ⚠️ Dotplot generation failed: {e}")

    # 5) Heatmap (cluster mean expression; prefer raw)
    if GENERATE_HEATMAP and len(available_markers) > 0:
        log_msg("\n   Generating expression heatmap...")
        try:
            cluster_expr = pd.DataFrame()
            for gene in available_markers:
                if (adata.raw is not None) and (gene in adata.raw.var_names):
                    Xg = adata.raw[:, gene].X
                elif gene in adata.var_names:
                    Xg = adata[:, gene].X
                else:
                    continue
                Xg = Xg.toarray().ravel() if hasattr(Xg, 'toarray') else np.asarray(Xg).ravel()
                cluster_expr[gene] = Xg
            cluster_expr['leiden_bbknn'] = adata.obs['leiden_bbknn'].values
            cluster_mean_expr = cluster_expr.groupby('leiden_bbknn').mean()
            plt.figure(figsize=(max(20, len(available_markers)*0.3), 10))
            sns.heatmap(
                cluster_mean_expr.T,
                cmap='RdYlBu_r',
                center=0,
                robust=True,
                yticklabels=True,
                xticklabels=True,
                cbar_kws={'label': 'Mean Expression'}
            )
            plt.title('Epithelial Marker Expression Across Clusters')
            plt.xlabel('Cluster')
            plt.ylabel('Marker Genes')
            plt.tight_layout()
            plt.savefig(fig_dir / f'heatmap_marker_expression.{FIGURE_FORMAT}', dpi=DPI, bbox_inches='tight')
            plt.close()
            log_msg(f"   ✓ Heatmap saved")
        except Exception as e:
            log_msg(f"   ⚠️ Heatmap generation failed: {e}")

    # 6) Batch facets
    if GENERATE_FACET_PLOTS and BATCH_KEY in adata.obs.columns:
        log_msg("\n   Generating batch-wise UMAP...")
        try:
            batches = sorted(adata.obs[BATCH_KEY].unique())
            n_batches = len(batches)
            n_cols = min(3, n_batches)
            n_rows = (n_batches + n_cols - 1) // n_cols
            
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 5*n_rows))
            axes = [axes] if n_batches == 1 else axes.flatten()
            
            for i, batch in enumerate(batches):
                adata_batch = adata[adata.obs[BATCH_KEY] == batch].copy()
                
                sc.pl.umap(
                    adata_batch,
                    color='leiden_bbknn',
                    ax=axes[i],
                    show=False,
                    title=f'Batch: {batch}',
                    legend_loc='right margin',
                    legend_fontsize=6
                )
            
            for i in range(n_batches, len(axes)):
                axes[i].axis('off')
            
            plt.tight_layout()
            plt.savefig(fig_dir / f'umap_by_batch.{FIGURE_FORMAT}', dpi=DPI, bbox_inches='tight')
            plt.close()
            log_msg(f"   ✓ Batch-wise UMAP saved")
        except Exception as e:
            log_msg(f"   ⚠️ Batch-wise UMAP generation failed: {e}")

    log_msg(f"\n   All figures saved to: {fig_dir}")

In [ ]:
def generate_summary_report(adata, available_markers, output_dir, processing_time=None):
    """Generate plain text summary report."""
    from datetime import datetime
    log_step(8, "Generate Summary Report")
    report = []
    report.append("="*70)
    report.append("Epithelial Cell Analysis - BBKNN Integration Summary")
    report.append("="*70)
    report.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    if processing_time:
        report.append(f"Processing time: {processing_time:.1f} seconds ({processing_time/60:.1f} minutes)")
    report.append("")
    report.append("[ Data Overview ]")
    report.append(f"  Cells: {adata.n_obs:,}")
    report.append(f"  Genes: {adata.n_vars:,}")
    report.append("")
    if BATCH_KEY in adata.obs.columns:
        report.append(f"[ Batch Distribution (key: {BATCH_KEY}) ]")
        batch_counts = adata.obs[BATCH_KEY].value_counts().sort_index()
        for batch, count in batch_counts.items():
            pct = count / adata.n_obs * 100
            report.append(f"  {batch}: {count:,} cells ({pct:.1f}%)")
        report.append("")
    report.append("[ Epithelial Marker Genes ]")
    report.append(f"  Total markers: {len(MARKER_EPITHELIAL)}")
    report.append(f"  Available: {len(available_markers)} ({len(available_markers)/len(MARKER_EPITHELIAL)*100:.1f}%)")
    report.append("")
    if RUN_CLUSTERING:
        report.append("[ Multi-resolution Clustering Results ]")
        for res in LEIDEN_RESOLUTIONS:
            key = f'leiden_bbknn_res{res}'
            if key in adata.obs.columns:
                n_clusters = adata.obs[key].nunique()
                default_marker = " (default)" if res == DEFAULT_RESOLUTION else ""
                report.append(f"  Resolution {res}: {n_clusters} clusters{default_marker}")
        report.append("")
    report.append("[ BBKNN Configuration ]")
    report.append(f"  neighbors_within_batch: {BBKNN_NEIGHBORS_WITHIN_BATCH}")
    report.append(f"  n_pcs: {BBKNN_N_PCS}")
    report.append(f"  batch_key: {BATCH_KEY}")
    report.append(f"  High variable genes: {USE_HVG} (n={N_TOP_GENES if USE_HVG else 'N/A'})")
    report.append("")
    report.append("[ Output Files ]")
    report.append(f"  - Data: {Path(output_dir) / 'epithelial_bbknn_integrated.h5ad'}")
    report.append(f"  - Figures: {Path(output_dir) / 'figures/'}*.{FIGURE_FORMAT}")
    if RUN_FIND_MARKERS:
        report.append(f"  - Markers: {Path(output_dir) / 'cluster_markers.csv'}")
    report.append("")
    report.append("="*70)
    report.append("Analysis completed successfully")
    report.append("="*70)
    report_text = '\n'.join(report)
    report_path = Path(output_dir) / "analysis_summary.txt"
    with open(report_path, 'w') as f:
        f.write(report_text)
    log_msg(f"\nReport saved to: {report_path}")
    log_msg("\n" + report_text)

## Main Analysis Pipeline

In [ ]:
def main():
    """Main entry: execute load → check → preprocess → PCA → BBKNN → UMAP/clustering → differential → visualization → report in sequence."""
    print("\n" + "="*70)
    print("Epithelial Cell BBKNN Integration Pipeline (Optimized v1.3)")
    print("="*70)

    start_time = time.time()

    log_msg("\nChecking Python environment...")
    try:
        import scanpy as sc
        from bbknn import bbknn as bbknn_func
        log_msg(f"   scanpy: {sc.__version__}")
        log_msg(f"   bbknn: installed")
    except ImportError as e:
        print(f"\nError: {e}", file=sys.stderr)
        print("\nInstallation: pip install scanpy bbknn", file=sys.stderr)
        sys.exit(1)

    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)
    log_msg(f"\nOutput directory: {output_dir}")

    # Step 1: Load
    adata = load_anndata(INPUT_H5AD_PATH)

    # Step 2: Marker check
    available_markers = check_marker_genes(adata)

    # Step 3: Preprocessing (don't overwrite .raw directly, full matrix saved in returned adata_full)
    adata_full, adata_hvg = preprocess_for_bbknn(
        adata,
        n_hvg=N_TOP_GENES,
        batch_key=BATCH_KEY,
        FORCE_INCLUDE_MARKERS=FORCE_INCLUDE_MARKERS
    )

    # Step 3.5: PCA
    adata_hvg = run_pca(adata_hvg, n_pcs=N_PCS)

    # Step 4: BBKNN integration
    adata_hvg = run_bbknn_integration(
        adata_hvg,
        batch_key=BATCH_KEY,
        neighbors_within_batch=BBKNN_NEIGHBORS_WITHIN_BATCH,
        n_pcs=BBKNN_N_PCS
    )

    # Step 5: UMAP and clustering
    if RUN_UMAP or RUN_CLUSTERING:
        adata_hvg = run_umap_and_clustering(
            adata_hvg,
            run_umap=RUN_UMAP,
            min_dist=UMAP_MIN_DIST,
            run_clustering=RUN_CLUSTERING,
            resolutions=LEIDEN_RESOLUTIONS,
            default_resolution=DEFAULT_RESOLUTION
        )

    # Step 6: Differential genes
    all_markers = None
    if RUN_FIND_MARKERS:
        all_markers = find_all_markers_optimized(
            adata_hvg,
            'leiden_bbknn',
            output_dir,
            min_pct=MARKER_MIN_PCT,
            logfc_threshold=MARKER_LOGFC_THRESHOLD
        )

    # Step 7: Visualization
    if RUN_UMAP:
        generate_visualizations(adata_hvg, available_markers, output_dir)

    # Backfill results to full object (for unified saving/downstream use)
    log_msg("\nBackfilling results to full dataset...")
    for key in ['X_pca','X_umap']:
        if key in adata_hvg.obsm:
            adata_full.obsm[key] = adata_hvg.obsm[key]
    for col in adata_hvg.obs.columns:
        if col.startswith('leiden_bbknn'):
            adata_full.obs[col] = adata_hvg.obs[col]
    if 'neighbors' in adata_hvg.uns:
        adata_full.uns['neighbors'] = adata_hvg.uns['neighbors']
    if 'connectivities' in adata_hvg.obsp:
        adata_full.obsp['connectivities'] = adata_hvg.obsp['connectivities']
    if 'distances' in adata_hvg.obsp:
        adata_full.obsp['distances'] = adata_hvg.obsp['distances']

    # Save integrated results
    log_msg("\nSaving integrated AnnData...")
    final_path = output_dir / "epithelial_bbknn_integrated.h5ad"
    log_msg("   Using gzip compression...")
    adata_full.write_h5ad(final_path, compression='gzip', compression_opts=9)
    file_size = final_path.stat().st_size / (1024**3)
    log_msg(f"   Saved: {final_path} ({file_size:.2f} GB)")

    # Report
    total_time = time.time() - start_time
    generate_summary_report(adata_full, available_markers, output_dir, total_time)

    # Terminal summary
    print("\n" + "="*70)
    print("✓ All analyses completed successfully")
    print("="*70)
    print(f"\nOutput directory: {output_dir}")
    print(f"Data: {final_path} ({file_size:.2f} GB)")
    print(f"Figures: {output_dir / 'figures/'}*.{FIGURE_FORMAT}")
    if RUN_FIND_MARKERS:
        print(f"Markers: {output_dir / 'cluster_markers.csv'}")
    print(f"\nTotal time: {total_time:.1f} seconds ({total_time/60:.1f} minutes)")

    print(f"\n📊 Key Information:")
    print(f"   Cells: {adata_full.n_obs:,}")
    print(f"   Available markers: {len(available_markers)}/{len(MARKER_EPITHELIAL)}")
    print(f"   Default clusters: {adata_full.obs['leiden_bbknn'].nunique()}")

    print(f"\n🔧 BBKNN Parameters:")
    print(f"   neighbors_within_batch: {BBKNN_NEIGHBORS_WITHIN_BATCH}")
    print(f"   n_pcs: {BBKNN_N_PCS}")
    print(f"   HVG: {N_TOP_GENES}")
    print()

    return adata_full, output_dir

## Execute Pipeline

In [ ]:
if __name__ == "__main__":
    try:
        adata, output_dir = main()
    except KeyboardInterrupt:
        print("\n\n⚠️  User interrupted", file=sys.stderr)
        sys.exit(130)
    except Exception as e:
        print(f"\n✖ Error: {e}", file=sys.stderr)
        import traceback
        traceback.print_exc()
        sys.exit(1)